# Project title

Your name, the date.

**Summary:** write your three main findings here last of all.

## Questions

1. Which category earns the most money, and is it also the most ordered?
2. Which city brings the most revenue?
3. How does revenue change month by month?
4. Do discounts change how much people buy, or how happy they are?
5. How do customers pay?

## Data

Source, licence, and what each column means (the data dictionary).

## Imports and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

## Load and first look

Write down every problem you find.

In [ ]:
raw_orders = pd.read_csv("data/raw/orders.csv", parse_dates=["date"])
print(raw_orders.shape)
raw_orders.info()
raw_orders.head()

In [ ]:
print(raw_orders.isna().sum())
print("duplicate rows:", raw_orders.duplicated().sum())
print(raw_orders["city"].value_counts())
raw_orders.describe().round(1)

## Cleaning

| Problem | Decision | Why |
|---------|----------|-----|
| 8 repeated rows | drop them | the same order counted twice |
| write the rest | | |

In [ ]:
def clean_orders(raw):
    df = raw.copy()
    df = df.drop_duplicates()
    df["city"] = df["city"].str.strip().str.title()
    df["payment"] = df["payment"].fillna("Unknown")
    df = df[df["qty"] <= 20]
    df["amount"] = df["qty"] * df["price"] * (1 - df["discount"] / 100)
    df["month"] = df["date"].dt.strftime("%Y-%m")
    df["weekday"] = df["date"].dt.day_name()
    return df.reset_index(drop=True)

orders = clean_orders(raw_orders)
print("rows before:", len(raw_orders), "after:", len(orders))

In [ ]:
def check(df):
    assert df["order_id"].is_unique, "repeated order ids"
    assert df["qty"].between(1, 20).all(), "qty out of range"
    assert df["payment"].notna().all(), "blank payment"
    print("all checks passed")

check(orders)

## Analysis

### Question 1: which category earns the most?

In [ ]:
cat = orders.groupby("category").agg(orders=("order_id", "count"), revenue=("amount", "sum")).sort_values("revenue", ascending=False)
cat["revenue"].plot(kind="bar", title="Revenue by category")
plt.ylabel("Rupees")
plt.show()
cat

Write one sentence with a number. Then add a section for each of your other questions.

## Conclusions

**Key insights:** what you found, why it matters, what to do.

**Limits:** what this analysis cannot tell you.

**Next steps:** what you would do next.